# Week 17: Capstone protocol, packaging, configs, and logging

Month 5 turns two experiments into **one professional project**. This week is pure engineering and research design: the habits that make code usable by a team, which is exactly what employers look for.

| Session | Topic | Time |
|---|---|---|
| Day 1 | `PROTOCOL.md`: fix your questions, metrics and tests *before* the final runs | 2.5 h |
| Day 2 | Packaging: `pyproject.toml`, `pip install -e .`, the repo layout | 2 h |
| Day 3 | Config validation and logging | 2.5 h |
| Day 4 | An imaging CLI: `python -m medlab.imaging.run --config …` | 2.5 h |
| Review | Spaced recall | 0.5 h |

**Read:** *The Good Research Code Handbook* (goodresearch.dev), chapters "Setup", "Decoupling code" and "Testing".

### Done when
- [ ] `PROTOCOL.md` is committed **before** any test-set number from Weeks 18–19 exists (Git timestamps prove it).
- [ ] `pip install -e .` works, and `import medlab` works from any folder.
- [ ] A bad config fails fast with a clear error message.
- [ ] Both experiments run from the command line and write logs to `logs/`.

In [ ]:
import sys, importlib, subprocess
from pathlib import Path
for p in ["..", "../capstone"]:
    if str(Path(p).resolve()) not in sys.path:
        sys.path.insert(0, str(Path(p).resolve()))
CAP = Path("../capstone")

def run_tests(*paths):
    """Run pytest inside the capstone folder and return its exit code (0 = all passed)."""
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", *paths], cwd=CAP, capture_output=True, text=True)
    print(r.stdout[-2500:], r.stderr[-800:])
    return r.returncode

def fresh(module_name):
    """Import (or re-import after you edited it) a module from your capstone package."""
    importlib.invalidate_caches()          # notice files you created after the kernel started
    return importlib.reload(importlib.import_module(module_name))

---
# Day 1: Pre-register your protocol (2.5 h)

If you decide *after* seeing the test results which metric, which split or which comparison to report, you can make almost anything look significant. This is called "p-hacking" or "researcher degrees of freedom", and it's a major cause of irreproducible ML. The fix: **write the plan down first and commit it.**

### Exercise 17.1: `capstone/PROTOCOL.md`
Use exactly these `##` headings (the checker looks for them), and fill each one in with specifics:
- `## Research questions`: RQ-A (retrieval) and RQ-B (imaging), one sentence each.
- `## Hypotheses`: e.g. "H-A1: hybrid RRF improves nDCG@10 over BM25 on the NFCorpus test set."
- `## Data and splits`: which splits are used for what. Test is used **once**, in Weeks 18–19.
- `## Systems`: every system and baseline, with the config file that defines it.
- `## Metrics`: the primary metric per experiment (nDCG@10; AUROC) and the secondary ones.
- `## Statistical analysis`: the paired randomisation test across queries and Holm correction (A); a bootstrap CI over test images, mean ± std over 3 seeds, and a paired bootstrap of the AUROC difference (B).
- `## Decision rules`: what counts as "better" (significant *and* at least x points?).
- `## Limitations known in advance`: for example, incomplete relevance judgements, no patient-level split information, small 28×28 images, no external validation.
- `## Ethics`: public data only; **not clinically validated**; intended for research and education only.

In [ ]:
# Write PROTOCOL.md in VS Code, commit it, then run the checker.

In [ ]:
txt = (CAP / "PROTOCOL.md").read_text()
required = ["Research questions", "Hypotheses", "Data and splits", "Systems", "Metrics",
            "Statistical analysis", "Decision rules", "Limitations known in advance", "Ethics"]
missing = [h for h in required if f"## {h}" not in txt]
assert not missing, f"missing sections: {missing}"
assert "not clinically validated" in txt.lower()
print("✅ 17.1 passed. Now commit it: git add PROTOCOL.md && git commit -m 'Pre-register evaluation protocol'")

---
# Day 2: Packaging (2 h)

Until now, notebooks found `medlab` through `sys.path` tricks. A real project is an **installable package**. Create `capstone/pyproject.toml`:
```toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "medlab"
version = "0.1.0"
description = "Baselines vs learned models for biomedical retrieval and medical image classification (research only, not clinically validated)"
requires-python = ">=3.10"
dependencies = ["numpy", "pandas", "scikit-learn", "nltk", "pyyaml", "torch", "torchvision", "medmnist", "sentence-transformers"]

[project.optional-dependencies]
dev = ["pytest", "pytest-cov", "ruff"]

[tool.setuptools.packages.find]
include = ["medlab*"]
```
Then, in the capstone folder: `pip install -e ".[dev]"`. The `-e` means *editable*: your code changes take effect without reinstalling.

**Target layout** (move files around if you need to):
```
medsearch-lab/
├── medlab/            # the package: data, text, index, ranking, dense, fusion, metrics, stats, evaluate, imaging/
├── configs/           # one YAML per experiment
├── scripts/           # one-off entry points (tables, figures, reproduce.sh)
├── tests/
├── results/           # small JSON/CSV results (committed); large raw outputs are git-ignored
├── report/            # the research report and figures
├── PROTOCOL.md  DATASETS.md  ERROR_ANALYSIS.md  README.md  pyproject.toml
```

In [ ]:
# Create pyproject.toml, run pip install -e ".[dev]" in a terminal, then run the checker.

In [ ]:
import tomllib
meta = tomllib.loads((CAP / "pyproject.toml").read_text())
assert meta["project"]["name"] == "medlab"
deps = " ".join(meta["project"]["dependencies"])
for d in ["numpy", "torch", "pyyaml", "scikit-learn"]:
    assert d in deps, f"missing dependency {d}"
print("✅ 17.2 passed")

---
# Day 3: Fail fast with config validation, and log everything (2.5 h)

**A config typo shouldn't cost you a 2-hour GPU run.** Validate configs as soon as you load them, and fail with a message that says exactly what's wrong.

**`print` versus `logging`:** `logging` gives you timestamps and levels (DEBUG/INFO/WARNING/ERROR), writes to files as well as the console, and can be silenced or turned up without editing code. Every serious codebase uses it.

### Exercise 17.3: `medlab/config.py` and `medlab/logging_utils.py`
```python
# config.py
REQUIRED = {"retrieval": {"name", "system", "split"}, "imaging": {"name", "model", "epochs", "lr", "seeds", "split"}}
VALID_SYSTEMS = {"tfidf", "bm25", "dense", "hybrid_rrf"}
VALID_MODELS = {"small_cnn", "resnet_frozen", "resnet_finetune", "logreg"}
def load_config(path, kind) -> dict    # yaml.safe_load; raise ValueError naming missing keys or invalid values;
                                       # split must be "dev"/"test" (retrieval) or "val"/"test" (imaging)
# logging_utils.py
def get_logger(name, log_file=None) -> logging.Logger   # INFO level, format "%(asctime)s %(levelname)s %(name)s: %(message)s";
                                                        # add a FileHandler if log_file is given; don't add duplicate handlers on repeat calls
```
Write `tests/test_config.py` with at least 4 tests, using pytest's `tmp_path` fixture to write small YAML files. Then use `load_config` and `get_logger` in `medlab/evaluate.py` (logging to `logs/<name>.log`).

In [ ]:
# Implement config.py, logging_utils.py and tests, then run the checker.

In [ ]:
import yaml, logging
assert run_tests("tests/test_config.py") == 0
cfgmod, logmod = fresh("medlab.config"), fresh("medlab.logging_utils")
tmp = CAP / "configs" / "_tmp_bad.yaml"
tmp.write_text(yaml.safe_dump({"name": "x", "system": "bm25"}))
try:
    cfgmod.load_config(tmp, "retrieval"); raise AssertionError("missing 'split' should raise")
except ValueError as e:
    assert "split" in str(e)
tmp.write_text(yaml.safe_dump({"name": "x", "system": "bm52", "split": "test"}))
try:
    cfgmod.load_config(tmp, "retrieval"); raise AssertionError("invalid system should raise")
except ValueError as e:
    assert "bm52" in str(e)
tmp.unlink()
lg = logmod.get_logger("medlab.test"); lg2 = logmod.get_logger("medlab.test")
assert lg is lg2 and len(lg.handlers) == 1 and lg.level == logging.INFO
print("✅ 17.3 passed")

---
# Day 4: The imaging command-line runner (2.5 h)

### Exercise 17.4: `medlab/imaging/run.py`
```bash
python -m medlab.imaging.run --config configs/imaging_small_cnn.yaml
```
With a config like:
```yaml
name: small_cnn
model: small_cnn          # small_cnn | resnet_frozen | resnet_finetune | logreg
epochs: 8
lr: 0.001
batch_size: 64
seeds: [0, 1, 2]
split: val                # "test" only in Week 19, once
```
For each seed: set the seed, build the model, `fit` it (on `get_device()`), and predict on `split`. Save `results/imaging/<name>_<split>.json` containing the config, the per-seed AUROC, the mean and std, and **the per-seed predicted probabilities** (you'll need these for CIs and calibration in Week 19 without retraining). Log progress with `get_logger`. `logreg` uses scikit-learn on flattened pixels (seeds don't matter for it, but keep the same output format).

Create configs for all four models (`resnet_finetune` with `lr: 0.0001`).

In [ ]:
# Implement medlab/imaging/run.py and the four configs, then run the checker.

In [ ]:
import subprocess, sys, json, os, yaml
cfg = yaml.safe_load((CAP / "configs" / "imaging_small_cnn.yaml").read_text())
cfg.update({"name": "ci_smoke", "epochs": 1, "seeds": [0, 1]})
(CAP / "configs" / "_ci_smoke.yaml").write_text(yaml.safe_dump(cfg))
out = subprocess.run([sys.executable, "-m", "medlab.imaging.run", "--config", "configs/_ci_smoke.yaml"], cwd=CAP, capture_output=True, text=True)
print(out.stdout[-600:], out.stderr[-1500:])
assert out.returncode == 0
res = json.load(open(CAP / "results" / "imaging" / "ci_smoke_val.json"))
assert len(res["per_seed"]) == 2 and "mean" in res and len(res["per_seed"][0]["probs"]) > 0
for m in ["small_cnn", "resnet_frozen", "resnet_finetune", "logreg"]:
    assert (CAP / "configs" / f"imaging_{m}.yaml").exists()
(CAP / "configs" / "_ci_smoke.yaml").unlink()
print("✅ 17.4 passed")

---
# Review (30 min)
1. Why pre-register? Give a concrete example of how not doing it could inflate your results.
2. What does `pip install -e .` do, and why is it better than `sys.path.insert`?
3. When would you log at WARNING rather than INFO?
4. *(Week 8)* What's wrong with picking the best of 30 configs by test score?
5. *(Week 12)* Explain the paired randomisation test to a non-statistician.
6. *(Week 16)* Why save per-seed probabilities instead of only the AUROC?

✍️ *Your answer:* (double-click to edit)

